# Double fine-tune SIRIS-Lab/citation-parser-ENTITY

Continued fine-tune of `SIRIS-Lab/citation-parser-ENTITY` on the public SIRIS dataset merged with a focused set of non-APA references labeled from the app's own parse cache.

**Before running:** upload the `merged/` folder to this Colab runtime. Easiest path is to zip it locally (`merged.zip`), drag-and-drop into Colab's Files panel, and the next cell will extract it automatically. Alternatively, mount Google Drive and point `MERGED_DIR` at the folder path.

Runtime: T4 GPU is sufficient. Expected wall time: 10–25 minutes.

**Note on the pip warning:** you'll see a red `fsspec` conflict warning from `gcsfs`. This is harmless — `gcsfs` is pre-installed in Colab but is not used by the training code, and `datasets==3.1.0` works fine with the downgraded fsspec. Keep going.

In [ ]:
!pip install -q transformers==4.46.3 datasets==3.1.0 evaluate==0.4.3 seqeval==1.2.2 accelerate==1.1.1

In [ ]:
# Unzip merged.zip if it's present and the merged/ folder doesn't already exist.
# This lets you drag-and-drop a single zipped folder into Colab's Files panel
# instead of uploading dozens of tiny files.
import os
import zipfile
from pathlib import Path

if Path("merged.zip").exists() and not Path("merged").exists():
    print("Extracting merged.zip...")
    with zipfile.ZipFile("merged.zip", "r") as zf:
        zf.extractall(".")
    # Handle the case where the zip has a parent folder like 'merged/merged/...'
    if not Path("merged/label_maps.json").exists():
        # Find the directory that contains label_maps.json and symlink/rename it
        for candidate in Path(".").rglob("label_maps.json"):
            src = candidate.parent
            if src.name != "merged":
                print(f"Found dataset at {src}, renaming to merged/")
                if Path("merged").exists():
                    import shutil
                    shutil.rmtree("merged")
                src.rename("merged")
            break
    print("Done. Contents of merged/:")
    for p in sorted(Path("merged").iterdir()):
        print(f"  {p.name}")
elif Path("merged").exists():
    print("merged/ already extracted.")
else:
    print("!! Neither merged/ nor merged.zip found in the working directory.")
    print("!! Upload merged.zip via Colab's Files panel, or mount Drive.")

In [ ]:
import json
from pathlib import Path

import numpy as np
import torch
from datasets import load_from_disk
from transformers import (
    AutoConfig,
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
import evaluate

MODEL_ID = "SIRIS-Lab/citation-parser-ENTITY"
MERGED_DIR = Path("merged")  # change this if you uploaded elsewhere
OUTPUT_DIR = Path("finetuned")

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  device: {torch.cuda.get_device_name(0)}")

In [ ]:
ds = load_from_disk(str(MERGED_DIR))
print({split: len(ds[split]) for split in ds.keys()})

label_maps = json.loads((MERGED_DIR / "label_maps.json").read_text(encoding="utf-8"))
label2id = label_maps["label2id"]
id2label = {int(k): v for k, v in label_maps["id2label"].items()}
num_labels = len(id2label)
print(f"num_labels = {num_labels}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_and_align(batch):
    tokenized = tokenizer(
        batch["tokens"],
        is_split_into_words=True,
        truncation=True,
        max_length=512,
    )
    aligned = []
    for i, labels in enumerate(batch["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev_word = None
        out = []
        for word_idx in word_ids:
            if word_idx is None:
                out.append(-100)
            elif word_idx != prev_word:
                out.append(labels[word_idx])
            else:
                out.append(-100)
            prev_word = word_idx
        aligned.append(out)
    tokenized["labels"] = aligned
    return tokenized

tokenized = ds.map(
    tokenize_and_align,
    batched=True,
    remove_columns=[c for c in ds["train"].column_names if c in {"tokens", "ner_tags", "id"}],
)
print({k: len(v) for k, v in tokenized.items()})

In [ ]:
config = AutoConfig.from_pretrained(MODEL_ID, num_labels=num_labels, id2label=id2label, label2id=label2id)
model = AutoModelForTokenClassification.from_pretrained(MODEL_ID, config=config)

data_collator = DataCollatorForTokenClassification(tokenizer)
seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    true_preds, true_labels = [], []
    for pred_row, label_row in zip(predictions, labels):
        preds_row, labs_row = [], []
        for p, l in zip(pred_row, label_row):
            if l == -100:
                continue
            preds_row.append(id2label[int(p)])
            labs_row.append(id2label[int(l)])
        true_preds.append(preds_row)
        true_labels.append(labs_row)
    results = seqeval_metric.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=5,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    weight_decay=0.01,
    warmup_ratio=0.1,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    seed=20260411,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

In [ ]:
print("=== eval on kaynaklar_test ===")
kaynaklar_metrics = trainer.evaluate(tokenized["kaynaklar_test"])
print(kaynaklar_metrics)

print("=== eval on public_test (catastrophic forgetting guard) ===")
public_metrics = trainer.evaluate(tokenized["public_test"])
print(public_metrics)

In [ ]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

summary = {
    "model_id": MODEL_ID,
    "kaynaklar_test": kaynaklar_metrics,
    "public_test": public_metrics,
}
(OUTPUT_DIR / "training_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(f"Saved to {OUTPUT_DIR}")
print("Zip it up and download:  !zip -r finetuned.zip finetuned")